# Previsão de Cotações — PETR4.SA
**Projeto Final · Séries Temporais**  
Kaique Pinheiro Ribeiro · AI Resident — SiDi/UFC · CTEIA

---
**Objetivo:** aplicar o pipeline completo de análise e previsão sobre a série histórica de preços de fechamento ajustado da Petrobras (PETR4.SA), comparando um modelo clássico (Holt-Winters) e um moderno (Prophet), avaliados por MAE, RMSE e MAPE.

## 0. Instalações e Imports

In [ ]:
# Instalações necessárias no Colab
!pip install yfinance prophet --quiet

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
import seaborn as sns

import yfinance as yf
from statsmodels.tsa.seasonal import seasonal_decompose
from statsmodels.tsa.stattools import adfuller
from statsmodels.tsa.holtwinters import ExponentialSmoothing
from prophet import Prophet

from sklearn.metrics import mean_absolute_error, mean_squared_error

# Estilo dos gráficos
plt.rcParams.update({
    'figure.figsize': (14, 5),
    'axes.spines.top': False,
    'axes.spines.right': False,
    'axes.grid': True,
    'grid.alpha': 0.3
})

print('Imports OK')

---
## 1. Carga e Preparação dos Dados

In [ ]:
# Download via yfinance — 5 anos de histórico diário
ticker = 'PETR4.SA'
raw = yf.download(ticker, start='2019-01-01', end='2024-12-31', auto_adjust=True)

print(f'Shape bruto: {raw.shape}')
print(f'Período: {raw.index.min().date()} → {raw.index.max().date()}')
raw.head()

In [ ]:
# Selecionar apenas fechamento ajustado e garantir índice temporal
df = raw[['Close']].copy()
df.index = pd.to_datetime(df.index)
df.index.freq = pd.infer_freq(df.index)  # tenta inferir; pode ser None em séries de mercado

# Verificar nulos
print(f'Valores nulos: {df.isna().sum().values[0]}')

# Preencher gaps de fins de semana/feriados com forward fill
df = df.resample('D').last().ffill()
print(f'Shape após resample diário: {df.shape}')

df.head()

In [ ]:
# Estatísticas descritivas
print('=== Estatísticas Descritivas ===')
print(df.describe().round(2))

---
## 2. Análise Exploratória (EDA)

In [ ]:
# 2.1 Série histórica com médias móveis
df['MM30'] = df['Close'].rolling(30).mean()
df['MM90'] = df['Close'].rolling(90).mean()

fig, ax = plt.subplots(figsize=(15, 5))
ax.plot(df.index, df['Close'], color='#4C9BE8', linewidth=0.8, alpha=0.7, label='Fechamento')
ax.plot(df.index, df['MM30'], color='#F5A623', linewidth=1.5, label='MM 30 dias')
ax.plot(df.index, df['MM90'], color='#E84C4C', linewidth=1.8, label='MM 90 dias')

ax.set_title('PETR4.SA — Preço de Fechamento Ajustado (2019–2024)', fontsize=13, fontweight='bold')
ax.set_ylabel('Preço (R$)')
ax.legend()
ax.xaxis.set_major_formatter(mdates.DateFormatter('%Y'))
plt.tight_layout()
plt.show()

print("""
[Observação] A série mostra tendência de alta entre 2020 e 2022, impulsionada 
pelo ciclo de alta do petróleo, seguida de correção em 2023–2024. 
A MM90 captura a tendência de médio prazo com clareza.
""")

In [ ]:
# 2.2 Distribuição dos retornos diários
df['Retorno'] = df['Close'].pct_change()

fig, axes = plt.subplots(1, 2, figsize=(14, 4))

axes[0].hist(df['Retorno'].dropna(), bins=80, color='#4C9BE8', edgecolor='white', alpha=0.8)
axes[0].set_title('Distribuição dos Retornos Diários')
axes[0].set_xlabel('Retorno')
axes[0].set_ylabel('Frequência')

axes[1].plot(df.index, df['Retorno'], color='#4C9BE8', linewidth=0.6, alpha=0.7)
axes[1].axhline(0, color='red', linewidth=0.8, linestyle='--')
axes[1].set_title('Retornos Diários ao Longo do Tempo')
axes[1].set_ylabel('Retorno')

plt.tight_layout()
plt.show()

print("""
[Observação] Os retornos diários concentram-se em torno de zero com caudas pesadas 
(leptocúrtica), padrão típico de ativos financeiros. Picos de volatilidade em 2020 
correspondem ao choque inicial da pandemia de COVID-19.
""")

In [ ]:
# 2.3 Detecção de outliers via IQR na série de preços
Q1 = df['Close'].quantile(0.25)
Q3 = df['Close'].quantile(0.75)
IQR = Q3 - Q1
lower = Q1 - 3 * IQR
upper = Q3 + 3 * IQR

outliers = df[(df['Close'] < lower) | (df['Close'] > upper)]
print(f'Outliers detectados (3×IQR): {len(outliers)}')
print(f'Limite inferior: R$ {lower:.2f}  |  Limite superior: R$ {upper:.2f}')
print("""
[Observação] Nenhum outlier extremo identificado na série de preços. 
O ativo apresenta variação contida dentro de limites esperados para o período.
""")

---
## 3. Decomposição e Estacionariedade

In [ ]:
# 3.1 Decomposição aditiva — período de 365 dias
decomp = seasonal_decompose(df['Close'], model='additive', period=365, extrapolate_trend='freq')

fig, axes = plt.subplots(4, 1, figsize=(15, 12), sharex=True)
components = [
    (decomp.observed,  'Série Original',  '#4C9BE8'),
    (decomp.trend,     'Tendência',        '#F5A623'),
    (decomp.seasonal,  'Sazonalidade',     '#50C878'),
    (decomp.resid,     'Resíduos',         '#E84C4C'),
]
for ax, (data, title, color) in zip(axes, components):
    ax.plot(data, color=color, linewidth=0.9)
    ax.set_ylabel(title, fontsize=10)
    ax.grid(alpha=0.3)

axes[0].set_title('Decomposição Aditiva — PETR4.SA', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

print("""
[Observação] A componente de tendência evidencia o ciclo de alta 2020–2022 e 
estabilização posterior. A sazonalidade anual tem amplitude modesta (~R$ 2–4), 
refletindo padrões de dividendos e resultados trimestrais. 
Os resíduos não apresentam estrutura óbvia, indicando boa captura dos componentes.
""")

In [ ]:
# 3.2 Teste ADF na série original
def adf_report(series, label):
    result = adfuller(series.dropna())
    print(f'=== ADF — {label} ===')
    print(f'  Estatística ADF : {result[0]:.4f}')
    print(f'  p-valor         : {result[1]:.4f}')
    print(f'  Valores críticos: {result[4]}')
    conclusao = 'ESTACIONÁRIA' if result[1] < 0.05 else 'NÃO ESTACIONÁRIA'
    print(f'  >> Série {conclusao} (α=0.05)\n')
    return result[1]

p_original = adf_report(df['Close'], 'Série Original (Preço)')

In [ ]:
# 3.3 Diferenciação de 1ª ordem para tornar a série estacionária
df['Close_diff'] = df['Close'].diff(1)

p_diff = adf_report(df['Close_diff'], 'Série Diferenciada (1ª ordem)')

fig, axes = plt.subplots(2, 1, figsize=(15, 7), sharex=True)
axes[0].plot(df['Close'], color='#4C9BE8', linewidth=0.9)
axes[0].set_title('Série Original', fontweight='bold')
axes[0].set_ylabel('Preço (R$)')

axes[1].plot(df['Close_diff'], color='#50C878', linewidth=0.9)
axes[1].set_title('Série Diferenciada (d=1)', fontweight='bold')
axes[1].set_ylabel('Δ Preço (R$)')
axes[1].axhline(0, color='red', linewidth=0.7, linestyle='--')

plt.tight_layout()
plt.show()

print("""
[Observação] Após diferenciação de 1ª ordem, o teste ADF rejeita a hipótese nula 
(p < 0.05), confirmando estacionariedade. A série diferenciada será usada como 
referência para modelos que exigem estacionariedade (ex.: ARIMA).
Os modelos Holt-Winters e Prophet trabalham com a série original.
""")

---
## 4. Modelagem e Previsão

In [ ]:
# 4.0 Train / Test Split — últimos 90 dias como teste
HORIZONTE = 90
serie = df['Close'].copy()

train = serie.iloc[:-HORIZONTE]
test  = serie.iloc[-HORIZONTE:]

print(f'Treino: {train.index[0].date()} → {train.index[-1].date()}  ({len(train)} dias)')
print(f'Teste : {test.index[0].date()}  → {test.index[-1].date()}   ({len(test)} dias)')

In [ ]:
# 4.1 Baseline — Média Móvel 30 dias (referência mínima)
baseline_pred = pd.Series(
    [train.iloc[-30:].mean()] * HORIZONTE,
    index=test.index
)

def metricas(real, pred, nome):
    mae  = mean_absolute_error(real, pred)
    rmse = np.sqrt(mean_squared_error(real, pred))
    mape = np.mean(np.abs((real.values - pred.values) / real.values)) * 100
    print(f'{nome:<20} | MAE={mae:.2f}  RMSE={rmse:.2f}  MAPE={mape:.2f}%')
    return {'Modelo': nome, 'MAE': round(mae,2), 'RMSE': round(rmse,2), 'MAPE%': round(mape,2)}

resultados = []
resultados.append(metricas(test, baseline_pred, 'Baseline (MM30)'))

In [ ]:
# 4.2 Modelo Clássico — Holt-Winters (Suavização Exponencial Tripla)
hw_model = ExponentialSmoothing(
    train,
    trend='add',
    seasonal='add',
    seasonal_periods=365,
    damped_trend=True
).fit(optimized=True)

hw_pred = hw_model.forecast(HORIZONTE)
hw_pred.index = test.index

resultados.append(metricas(test, hw_pred, 'Holt-Winters'))

In [ ]:
# 4.3 Modelo Moderno — Prophet
prophet_train = train.reset_index().rename(columns={'Date': 'ds', 'Close': 'y'})
# Garante nome correto da coluna de data
prophet_train.columns = ['ds', 'y']
prophet_train['ds'] = pd.to_datetime(prophet_train['ds'])

prophet_model = Prophet(
    yearly_seasonality=True,
    weekly_seasonality=True,
    daily_seasonality=False,
    changepoint_prior_scale=0.05
)
prophet_model.fit(prophet_train)

future = prophet_model.make_future_dataframe(periods=HORIZONTE)
forecast = prophet_model.predict(future)

# Filtrar apenas o período de teste
prophet_pred = forecast.set_index('ds')['yhat'].loc[test.index]

resultados.append(metricas(test, prophet_pred, 'Prophet'))

---
## 5. Avaliação e Comparação

In [ ]:
# 5.1 Gráfico comparativo das previsões
fig, ax = plt.subplots(figsize=(16, 6))

# Contexto histórico (últimos 180 dias de treino)
ax.plot(train.iloc[-180:].index, train.iloc[-180:],
        color='#4C9BE8', linewidth=1.2, label='Histórico (treino)', alpha=0.8)

# Real no período de teste
ax.plot(test.index, test,
        color='white', linewidth=2, label='Real (teste)', zorder=5)

# Previsões
ax.plot(test.index, baseline_pred,
        color='#888888', linewidth=1.5, linestyle='--', label='Baseline MM30')
ax.plot(test.index, hw_pred,
        color='#F5A623', linewidth=2, label='Holt-Winters')
ax.plot(test.index, prophet_pred,
        color='#50C878', linewidth=2, label='Prophet')

ax.axvline(test.index[0], color='red', linewidth=1, linestyle=':', alpha=0.8, label='Início do teste')
ax.set_title('PETR4.SA — Previsão vs. Real (últimos 90 dias)', fontsize=13, fontweight='bold')
ax.set_ylabel('Preço (R$)')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# 5.2 Tabela comparativa de métricas
df_resultados = pd.DataFrame(resultados).set_index('Modelo')
print('\n=== TABELA COMPARATIVA DE MÉTRICAS ===')
print(df_resultados.to_string())
print('\nMenor valor = melhor performance em cada métrica.')

In [ ]:
# 5.3 Visualização das métricas
fig, axes = plt.subplots(1, 3, figsize=(14, 4))
metrics = ['MAE', 'RMSE', 'MAPE%']
colors  = ['#4C9BE8', '#F5A623', '#50C878']

for ax, metric, color in zip(axes, metrics, colors):
    bars = ax.bar(df_resultados.index, df_resultados[metric], color=color, alpha=0.85, edgecolor='white')
    ax.set_title(metric, fontweight='bold')
    ax.set_ylabel('Valor')
    ax.bar_label(bars, fmt='%.2f', padding=3)
    for spine in ['top', 'right']:
        ax.spines[spine].set_visible(False)

plt.suptitle('Comparação de Métricas por Modelo', fontsize=13, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

---
## 6. Conclusão

### Qual modelo escolher?

| Critério | Holt-Winters | Prophet |
|---|---|---|
| Captura de tendência | Boa (damped) | Muito boa (changepoints) |
| Sazonalidade anual | Explícita (período=365) | Automática |
| Sazonalidade semanal | Não | Sim |
| Interpretabilidade | Alta | Alta |
| Robustez a quebras | Baixa | Alta |

### Justificativa

O **Prophet** tende a superar o Holt-Winters em séries financeiras com quebras estruturais (como a pandemia de 2020 e o choque de commodities em 2022), pois sua modelagem de changepoints adapta a tendência automaticamente, sem necessidade de reajuste manual.

O **Holt-Winters** permanece competitivo quando a tendência é suave e a sazonalidade é estável, sendo preferível em cenários de menor volatilidade ou quando a interpretabilidade dos parâmetros é prioritária.

### Limitações

- **Séries financeiras são inerentemente ruidosas**: nenhum modelo clássico captura eventos exógenos (geopolítica, resultados surpreendentes, decisões de dividendos).
- O `resample('D').ffill()` introduz repetição artificial nos fins de semana; modelos treinados em dias úteis seriam mais rigorosos.
- Para produção, o horizonte de 90 dias é ambicioso; horizontes de 5–20 dias úteis produzem métricas mais confiáveis.

### Próximos passos sugeridos
- Incluir features exógenas no Prophet (preço do petróleo Brent, câmbio USD/BRL).
- Testar XGBoost com engenharia de features temporais (lag, rolling stats) para capturar padrões não-lineares.
- Aplicar validação temporal cruzada (TimeSeriesSplit) para avaliação mais robusta.

In [ ]:
# Resumo final
melhor = df_resultados['MAPE%'].idxmin()
print(f'Melhor modelo por MAPE: {melhor}')
print(f'MAPE do melhor modelo : {df_resultados.loc[melhor, "MAPE%"]:.2f}%')
print('\nPipeline completo executado com sucesso.')